In [2]:
from dotenv import load_dotenv
load_dotenv()

import os
import re
import json
from pathlib import Path
from langchain_core.documents import Document
from datetime import datetime

# 1. 定义相对路径
RELATIVE_INDEX_PATH = os.path.join("..", "data", "chat-history-2605", "index.html")
INDEX_PATH = Path(RELATIVE_INDEX_PATH)

print(f"📂 正在读取包含 JS 数据的 HTML 文件: {INDEX_PATH}")

if not INDEX_PATH.exists():
    raise FileNotFoundError(f"❌ 未找到文件，请检查路径。当前路径: {INDEX_PATH.resolve()}")

html_content = INDEX_PATH.read_text(encoding="utf-8")

# 2. 使用正则表达式提取隐藏在 JS 中的所有 JSON 对象
json_pattern = re.compile(r'\{"type":\s*\d+.*?\}')
json_matches = json_pattern.findall(html_content)

print(f"🔍 从文件中成功提取到 {len(json_matches)} 条原始聊天记录。")


# 3. 解析 JSON 并转换为富含多媒体信息的标准文本，同时暂存时间戳
parsed_messages = []

for raw_json in json_matches:
    try:
        msg = json.loads(raw_json)
        msg_type = msg.get("type")
        msg_timestamp = msg.get("timestamp")
        msg_subtype = msg.get("sub_type")
        msg_url = msg.get("url")
        msg_title = msg.get("title")
        is_send = msg.get("is_send", 0)
        
        # 角色转换
        sender = "聪明姐姐" if is_send == 1 else "小郑猫猫"
        
        # 提取核心内容
        content_text = msg.get("text", "").strip()

        # 时间戳解析
        time_str = ""
        if msg_timestamp:
            time_str = datetime.fromtimestamp(msg_timestamp).strftime('[%Y-%m-%d %H:%M:%S] ')
        
        # 处理不同消息类型的文本映射
        if msg_type == 1:
            body = f"{sender}: {content_text}"
        elif msg_type == 43:
            body = f"{sender} 发送了视频: 【视频内容: {content_text}】"
        elif msg_type == 47:
            body = f"{sender} 发送了表情包: 【表情包内容: {content_text}】"
        elif msg_type == 49:
            if msg_subtype == 36:
                body = f"{sender} 发送了微信小程序链接: 【微信小程序链接内容: {msg_url}，微信小程序链接标题: {msg_title}】"
            elif msg_subtype == 5:
                body = f"{sender} 发送了普通链接: 【普通链接内容: {msg_url}，普通链接标题: {msg_title}】"
            else:
                body = f"{sender}: {content_text}"
        elif msg_type == 34:
            body = f"{sender} 发送了语音: 【语音内容: {content_text}】"
        elif msg_type == 3:
            body = f"{sender} 发送了图片: 【图片内容: {content_text}】"
        else:
            body = f"{sender}: {content_text}"
        
        formatted_line = f"{time_str}{body}" 
        
        # 将格式化后的文本连同原始时间戳一起存入临时列表，供切片使用
        parsed_messages.append({
            "text": formatted_line,
            "timestamp": msg_timestamp
        })
        
    except Exception as e:
        continue

print(f"📊 成功格式化 {len(parsed_messages)} 条聊天记录，准备进行智能切片...")


# 4. 基于 5 分钟静默期（300秒）的会话滑动窗口切片
SILENCE_THRESHOLD = 300  # 5分钟静默期
MAX_CHAT_COUNT = 30      # 单包对话消息上限兜底，防止单个 Chunk 过长

html_chunks = []
current_session_lines = []
current_session_start_idx = 0
last_timestamp = None

for idx, msg in enumerate(parsed_messages):
    current_time = msg["timestamp"]
    
    # 触发切片断开条件：非第一条消息，且（静默超5分钟 或 当前块攒满了30条）
    if last_timestamp is not None and current_time:
        time_gap = current_time - last_timestamp
        
        if (time_gap > SILENCE_THRESHOLD) or (len(current_session_lines) >= MAX_CHAT_COUNT):
            chunk_content = "\n".join(current_session_lines)
            
            if chunk_content.strip():
                doc = Document(
                    page_content=chunk_content,
                    metadata={
                        "source": RELATIVE_INDEX_PATH,
                        "file_type": "chat_text",
                        "session_start_time": parsed_messages[current_session_start_idx]["timestamp"],
                        "session_end_time": last_timestamp,
                        "message_count": len(current_session_lines)
                    }
                )
                html_chunks.append(doc)
            
            # 重置窗口状态
            current_session_lines = []
            current_session_start_idx = idx

    current_session_lines.append(msg["text"])
    last_timestamp = current_time

# 收尾：处理最后未闭合的对话块
if current_session_lines and "\n".join(current_session_lines).strip():
    doc = Document(
        page_content="\n".join(current_session_lines),
        metadata={
            "source": RELATIVE_INDEX_PATH,
            "file_type": "chat_text",
            "session_start_time": parsed_messages[current_session_start_idx]["timestamp"],
            "session_end_time": last_timestamp,
            "message_count": len(current_session_lines)
        }
    )
    html_chunks.append(doc)

print(f"\n============ 🚀 智能时间切片完成 ============")
print(f"💾 基于5分钟密集互动切分，最终生成 RAG 知识切片 (Chunks): {len(html_chunks)} 个")
print(f"=============================================")

📂 正在读取包含 JS 数据的 HTML 文件: ..\data\chat-history-2605\index.html
🔍 从文件中成功提取到 50395 条原始聊天记录。
📊 成功格式化 50100 条聊天记录，准备进行智能切片...

============ 🚀 智能时间切片完成 ============
💾 基于5分钟密集互动切分，最终生成 RAG 知识切片 (Chunks): 4656 个


In [3]:
print("--- [测试一：5分钟高频会话切片抽样验证] ---")

if html_chunks:
    for i, chunk in enumerate(html_chunks[:5]):
        start_dt = datetime.fromtimestamp(chunk.metadata['session_start_time']).strftime('%Y-%m-%d %H:%M:%S')
        end_dt = datetime.fromtimestamp(chunk.metadata['session_end_time']).strftime('%Y-%m-%d %H:%M:%S')
        duration = chunk.metadata['session_end_time'] - chunk.metadata['session_start_time']
        
        print(f"[会话块 {i+1}]")
        print(f"[{start_dt} 至 {end_dt} (持续发言 {duration} 秒)]")
        print(f"{chunk.page_content}")
        print("=" * 60)
else:
    print("❌ 未成功生成任何切片，请确认提取到的聊天记录数量不为 0。")

--- [测试一：5分钟高频会话切片抽样验证] ---
[会话块 1]
[2025-05-01 15:17:31 至 2025-05-01 15:17:31 (持续发言 0 秒)]
[2025-05-01 15:17:31] 小郑猫猫: 我通过了你的朋友验证请求，现在我们可以开始聊天了
[会话块 2]
[2025-05-25 16:57:00 至 2025-05-25 16:58:46 (持续发言 106 秒)]
[2025-05-25 16:57:00] 聪明姐姐: 哪里吃小龙虾[色]
[2025-05-25 16:58:42] 小郑猫猫: 我们家楼下
[2025-05-25 16:58:46] 小郑猫猫: 民乐菜馆
[会话块 3]
[2025-05-25 17:20:53 至 2025-05-25 17:26:10 (持续发言 317 秒)]
[2025-05-25 17:20:53] 聪明姐姐 发送了视频: 【视频内容: ./video/2025-05/20250525172053.mp4】
[2025-05-25 17:21:26] 聪明姐姐 发送了视频: 【视频内容: ./video/2025-05/20250525172126.mp4】
[2025-05-25 17:24:47] 小郑猫猫: 坏了
[2025-05-25 17:25:03] 小郑猫猫 发送了表情包: 【表情包内容: http://wxapp.tc.qq.com/262/20304/stodownload?m=1fe7038785a8d5569b87f378c3178c65&filekey=30350201010421301f020201060402534804101fe7038785a8d5569b87f378c3178c65020300af4c040d00000004627466730000000132&hy=SH&storeid=26306fe2d000a2516000000000000010600004f5053481bc64b40b68dcaee8&bizid=1023】
[2025-05-25 17:25:35] 聪明姐姐: 太牛了
[2025-05-25 17:25:41] 聪明姐姐: 百灵鸟
[2025-05-25 17:26:02] 小郑猫猫: [撇嘴]还是孙老师厉害
[

In [4]:
import os
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from tqdm import tqdm

# ==========================================
# 1. 修改为你本地已有或新下载的 bge-base-zh-v1.5 模型路径
# ==========================================
# 提示：你可以去 HuggingFace 或 ModelScope(魔搭社区) 下载 bge-base-zh-v1.5 到这个目录下
LOCAL_MODEL_PATH = r"E:\workspace\loverAgent\models\bge-base-zh-v1.5"

print(f"🚀 正在从本地绝对路径加载 bge-base-zh-v1.5 模型:\n👉 {LOCAL_MODEL_PATH}")

if not os.path.exists(LOCAL_MODEL_PATH):
    raise FileNotFoundError(
        f"❌ 未在指定路径找到模型，请核对路径是否正确！\n"
        f"当前尝试读取的路径: {os.path.abspath(LOCAL_MODEL_PATH)}\n"
        f"💡 请确保已将 BAAI/bge-base-zh-v1.5 下载解压至该文件夹。"
    )

# 2. 配置本地模型的运行参数
model_kwargs = {'device': 'cpu'}               # 如果电脑有英伟达显卡，可以改成 'cuda' 加速
encode_kwargs = {'normalize_embeddings': True}  # 归一化向量

local_embeddings = HuggingFaceEmbeddings(
    model_name=LOCAL_MODEL_PATH,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 3. 定义本地向量数据库的持久化路径
# 💡 提示：因为更换了 Embedding 模型，向量维度从 M3 的 1024 维变为了 base 的 768 维。
# 强烈建议换一个新的数据库目录（例如增加 _base 后缀），否则老数据库会因为维度不匹配报错。
CHROMA_DB_DIR = "./chroma_db_local_base"

print(f"\n📦 模型加载成功！正在将聊天记录与多媒体锚点注入本地向量库: {CHROMA_DB_DIR}")

# 4. 初始化一个空的 Chroma 向量库
vector_store = Chroma(
    embedding_function=local_embeddings,
    persist_directory=CHROMA_DB_DIR
)

# 5. 分批次（Batch）写入并显示进度条
# 💡 因为 bge-base 模型非常轻量，CPU 负担小，这里将 BATCH_SIZE 扩大到 128 提升吞吐量
BATCH_SIZE = 128  
total_chunks = len(html_chunks)

print(f"📊 开始分批次向量化入库，总切片数: {total_chunks}，每批大小: {BATCH_SIZE}")

# 使用 tqdm 展现进度条
with tqdm(total=total_chunks, desc="向量化进度", unit="chunk") as pbar:
    for i in range(0, total_chunks, BATCH_SIZE):
        batch = html_chunks[i : i + BATCH_SIZE]
        # 将当前批次加入向量库
        vector_store.add_documents(documents=batch)
        # 更新进度条
        pbar.update(len(batch))

print(f"\n✅ 向量化并入库成功！整个过程完全离线完成。")

🚀 正在从本地绝对路径加载 bge-base-zh-v1.5 模型:
👉 E:\workspace\loverAgent\models\bge-base-zh-v1.5

📦 模型加载成功！正在将聊天记录与多媒体锚点注入本地向量库: ./chroma_db_local_base
📊 开始分批次向量化入库，总切片数: 4656，每批大小: 128


向量化进度:   0%|          | 0/4656 [02:22<?, ?chunk/s]


RuntimeError: Numpy is not available

: 

In [ ]:
# ==============================================================================
# --- [测试二：本地向量库与多媒体留存验证] ---
# ==============================================================================
print("\n" + "="*20 + " [测试二：本地向量库与多媒体留存验证] " + "="*20)

# 1. 检查向量数据库中的总记录数
try:
    collection_count = vector_store._collection.count()
    print(f"📊 当前本地向量数据库中实际存储的记录总数: {collection_count} 条")

    # 验证数量是否与当前上下文中的切片数完全对齐
    if 'html_chunks' in locals() or 'html_chunks' in globals():
        if collection_count == len(html_chunks):
            print("✅ 数量验证通过：所有本地聊天切片已 100% 写入数据库。")
        else:
            print(f"⚠️ 数量不匹配！当前内存切片数 ({len(html_chunks)}) 与向量库记录数 ({collection_count}) 不一致，可能存在历史积压数据。")
    else:
        print("ℹ️ 提示：独立运行测试脚本中，已跳过与内存 html_chunks 的数量对比。")
except Exception as e:
    print(f"❌ 数据库读取失败，请检查 Chroma 连接: {e}")


# 2. 模拟多媒体语义检索压力测试
# 定义一组高频、模糊的多媒体意图测试集，全面压测本地 bge-base-zh-v1.5 的双向召回能力
test_cases = [
    {"query": "找一下聊天记录里的视频文件或者录像", "desc": "视频类目模糊测试"},
    {"query": "发给我的照片、图片或者截图在哪", "desc": "图片类目模糊测试"},
    {"query": "聊到关于去看电影、吃晚饭或者点外卖的内容了吗", "desc": "生活场景语义泛化测试"},
    {"query": "最长的一次聊天是多久", "desc": "元数据测试"}
]

print(f"\n🚀 开始执行多媒体语义检索压力测试（共 {len(test_cases)} 组用例）...")

cases_len = len(test_cases)

for idx, case in enumerate(test_cases, 1):
    query = case["query"]
    print(f"\n🔍 [压力测试 用例 {idx}/{cases_len}] 需求: '{query}' ({case['desc']})")
    
    try:
        # 使用 similarity_search_with_score 获取精确的 L2 距离分数（分值越低越接近 0，代表越相似）
        test_search_results = vector_store.similarity_search_with_score(query, k=1)
        
        if test_search_results:
            doc, score = test_search_results[0]
            print(f"🎯 命中 Top 1 记录！[L2 距离分数]: {score:.4f} (💡 越接近 0 越相关)")
            print("-" * 60)
            print(doc.page_content.strip())
            print("-" * 60)
            
            # 自动验证召回的文本里是否真的含有静态资源特征
            content_lower = doc.page_content.lower()
            if any(ext in content_lower for ext in ['.mp4', '.mkv', '.png', '.jpg', '.jpeg', 'src=', 'href=']):
                print("✅ 语义对齐成功：本地模型成功定位并穿透到多媒体静态资源路径！")
            else:
                print("ℹ️ 命中上下文：未能直接匹配文件后缀，但命中该语境下的文本对话。")
        else:
            print("⚠️ 未检索到任何相关内容，可能你的知识库中暂无相关语境。")
            
    except Exception as e:
        print(f"❌ 本地检索测试失败，报错信息: {e}")

print("\n" + "="*25 + " 测试流程全部结束 " + "="*25)s

In [ ]:
import os
from datetime import datetime

# 1. 确保向量库已经正确加载（如果你是在同一个 Notebook 会话中，直接沿用 vector_store 即可）
# 如果你是单独启动的该单元格，它会自动读取刚才本地持久化的 "./chroma_db_local_base" 目录
print("🔍 正在初始化本地检索器...")

# 2. 将向量库转换为检索器 (Retriever)
# search_type="similarity" 表示使用标准的余弦相似度检索
# search_kwargs={"k": 3} 表示每次用户提问，精准召回最相关的 3 个密集对话块（Chunks）
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)

print("✅ 检索器初始化成功！已准备好捕获聊天记录与多媒体资产。")

In [8]:
import sys
import os

# 引用 llm
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)
from llm.eval_llm import eval_chain

print("--- [测试三：本地多媒体检索与召回深度验证 + LLM 裁判审计] ---")

# 辅助函数：格式化时间戳输出
def format_time(ts):
    if ts:
        return datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
    return "未知时间"

# 测试问题
test_queries = [
    "找一下聊天记录里的视频文件",          # 模糊测试
    "哪里有吃小龙虾的推荐吗？",             # 关联测试
    "把发过的图片或微信小程序链接搜出来"      # 复合多媒体测试
]

for q_idx, query in enumerate(test_queries):
    print(f"\n🚀 【检索实验 {q_idx + 1}】 用户提问: '{query}'")
    
    # 1. 执行召回
    retrieved_docs = retriever.invoke(query)
    print(f"📊 成功召回最相关的 [ {len(retrieved_docs)} ] 个对话块：")
    print("-" * 70)
    
    query_total_score = 0
    valid_doc_count = len(retrieved_docs)
    
    # 2. 遍历召回文档并进行大模型审计
    for doc_idx, doc in enumerate(retrieved_docs):
        meta = doc.metadata
        start_time_str = format_time(meta.get("session_start_time"))
        end_time_str = format_time(meta.get("session_end_time"))
        content = doc.page_content
        
        print(f"👉 匹配片段 {doc_idx + 1} | 会话波峰: {start_time_str} ~ {end_time_str} | 包含消息: {meta.get('message_count', 0)} 条")
        
        # 深度高亮检查
        if any(tag in content for tag in ["【视频内容:", "【图片内容:", "【表情包内容:"]):
            print("💡 [📢 发现多媒体静态资源特征！]")
            
        # --------------- 🤖 LLM 裁判介入评估 ---------------
        try:
            # 传入当前 Query 和单条召回的 Content
            eval_result = eval_chain.invoke({"query": query, "context": content})
            llm_score = eval_result.get("score", 1)
            llm_reason = eval_result.get("reason", "未能成功评估")
            
            query_total_score += llm_score
            
            # 根据得分打印不同的高亮颜色或标签
            score_flag = "✅ 优" if llm_score >= 4 else "⚠️ 弱" if llm_score == 3 else "❌ 差"
            print(f"🤖 [LLM 裁判打分]: {score_flag} {llm_score}分 | 原因: {llm_reason}")
            
        except Exception as e:
            print(f"⚠️ [裁判评估异常]: {e}")
        # --------------------------------------------------
        
        print(f"📜 文本上下文:\n{content[:200]}...") # 打印前200字即可，避免刷屏
        print("-" * 50)
    
    # 3. 输出当前 Query 的召回准确率统计
    if valid_doc_count > 0:
        avg_score = query_total_score / valid_doc_count
        # 将 1-5 分映射成百分制召回准确度
        accuracy_pct = ((avg_score - 1) / 4) * 100
        print(f"🎯 【实验 {q_idx + 1} 召回质量报告】: 平均得分 {avg_score:.2f} / 5.0 | 召回准确率（语义相关度）: {accuracy_pct:.1f}%")
    else:
        print("❌ 本次实验未能召回任何有效文档，召回率为 0%")
        
    print("=" * 70)

e:\workspace\loverAgent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- [测试三：本地多媒体检索与召回深度验证 + LLM 裁判审计] ---

🚀 【检索实验 1】 用户提问: '找一下聊天记录里的视频文件'


NameError: name 'retriever' is not defined